# Training, weights and assumptions

This notebook refits the main model from the bundled audited historical training data. It reproduces the chronological prior, training blocks, score normalization and selected hyperparameters, then refits movement variance, feature coefficients, signed loadings and polling variance.

**Different weights have different meanings:** historical decay weights old cycles; tau regularizes feature coefficients; local multipliers control prior variance; lambda allocates signed covariance; blend weight translates a final mean. They are not interchangeable.

In [2]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p/'election_lab.py').exists())
sys.path.insert(0, str(ROOT))
import election_lab as lab
pd.set_option('display.max_rows', 40)

SCENARIO = 'matched_live'
YEAR = 2026
RUN_STUDENT = True
print(json.dumps(json.loads((ROOT/'MODEL_REGISTRY.json').read_text()),indent=2))

{
  "schema_version": 1,
  "main": "repaired_both__selected",
  "main_label": "Gaussian: repaired state priors + national features + signed state factor",
  "nonbayesian": "bias",
  "fat_tail": "both__t5v1",
  "fat_tail_note": "Research Student-t helper; original priors and common national factor, no signed factor. Not a Student version of the final main architecture.",
  "blend_weights": [
    0.05,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.7
  ],
  "blend_rule": "mean translation, Bayesian covariance unchanged",
  "default_blend_weight": null,
  "data_as_of": "2026-09-17",
  "year": 2026,
  "designation": "User-authorized standalone main; research registry left unchanged",
  "matched_fat_tail_experiment": "final_gaussian_matched_componentwise_t5",
  "matched_fat_tail_status": "Research comparison; all final Gaussian calibration held fixed",
  "older_gaussian": "both",
  "reference_not_universally_best": true,
  "ensemble_config": "config/ensemble_v1.json",
  "validation": "R

## 1. Refit the main model

Only cycles earlier than YEAR enter training. Selectors use their earlier-fold validation surfaces; the held-out outcome is not used. This reproduces the final chosen architecture rather than rerunning every abandoned research model. Model equations and hyperparameter grids are in [MODEL.md](../docs/MODEL.md).

In [3]:
RUN=lab.run_logged(lab.refit_main,SCENARIO,YEAR)
metadata=json.loads((RUN/'run.json').read_text())
print(json.dumps(metadata,indent=2))

Run log: cache/logs/refit_main_20260921T044211.608540Z.txt
{
  "scenario": "matched_live",
  "cycle": 2026,
  "training_cycles": [
    1978,
    1980,
    1982,
    1984,
    1986,
    1988,
    1990,
    1992,
    1994,
    1996,
    1998,
    2000,
    2002,
    2004,
    2006,
    2008,
    2010,
    2012,
    2014,
    2016,
    2018,
    2020,
    2022,
    2024
  ],
  "tau": 6.0,
  "signed_lambda": 0.5,
  "movement_kappa": 8.0,
  "poll_kappa": 2.0,
  "movement_iterations": 46,
  "passed": true,
  "selection_checks": {
    "feature_tau": "both_tau6",
    "signed_factor": "repaired_both__lambda0.5",
    "validation_cycles": [
      2020,
      2022,
      2024
    ]
  },
  "selection": "Reproduced earlier-cycle choices; not reselected on this test outcome"
}


## 2. Inspect learned coefficients and state dependence

The two national feature coefficients are shared; priors and variance budgets vary by state. Signed factor loadings can have opposite signs. A separate common national component can move states together. Empirical-Bayes variance/hyperparameter uncertainty remains fixed.

In [4]:
folds=pd.read_parquet(ROOT/'assets/main/folds.parquet')
r=folds[folds.scenario.eq(SCENARIO)&folds.cycle.eq(YEAR)].iloc[0]
f=np.load(ROOT/'assets/main'/r.fit_path)
display(pd.DataFrame({'feature':f['terms'],'coefficient_mean_pp':f['beta_mean'],'coefficient_sd_pp':np.sqrt(np.diag(f['beta_covariance'])),'current_standardized_input':f['z']}))
display(pd.DataFrame({'state':lab.gaussian.STATES,'prior_budget_sd_pp':np.sqrt(f['budget']),'signed_loading_pp':f['loading'],'prior_variance_multiplier':f['multipliers']}).round(3))

,feature,coefficient_mean_pp,coefficient_sd_pp,current_standardized_input
0,economy_momentum_wh,0.589440,1.280922,0.681189
1,approval_wh,3.084509,1.286158,1.602055


,state,prior_budget_sd_pp,signed_loading_pp,prior_variance_multiplier
0,AL,7.634,0.816,0.25
1,AK,8.245,-0.000,0.25
2,AZ,9.167,3.009,0.25
3,AR,8.179,0.945,0.25
4,CA,6.897,0.273,0.25
...,...,...,...,...
45,VA,7.593,-0.225,0.25
46,WA,7.076,-0.291,0.25
47,WV,20.523,-5.063,0.82
48,WI,7.492,1.555,0.25


## 3. Exact score recipe

Ingredients have fixed directions/weights, with training-only normalization. Current feature values cannot set the training centers, scales or imputation values. See the downloaded-source mapping and date/missingness policy in [DATA.md](../docs/DATA.md).

In [5]:
recipe=json.loads((ROOT/'config/fixed_feature_scores_v2.json').read_text())
print('Party orientation:',recipe['party_policy'])
print('Ingredient weights:',recipe['weights'])
print('Missing components:',recipe['missing_policy'])
print('Approval: (approval − 50)/10 before White-House-party orientation and final training scaling.')
display(pd.DataFrame(recipe['components']).T.query('score == "economy_momentum"'))

Party orientation: multiply economic levels, momentum and approval by +1 for Democratic WH / -1 for Republican WH; disruption stays unsigned
Ingredient weights: equal 1/3 sentiment, employment and cost pressure; cost families equal within block; change anchors equal within family
Missing components: all required components must be known; no imputation or renormalization
Approval: (approval − 50)/10 before White-House-party orientation and final training scaling.


,score,group,direction,weight,center
consumer_sentiment_3m__change_jan01,economy_momentum,sentiment,1,0.166667,zero
unemployment_pct__change_jan01,economy_momentum,employment,-1,0.166667,zero
inflation_yoy_pct__change_jan01,economy_momentum,cost_pressure,-1,0.055556,zero
cpi_level__pct_change_jan01,economy_momentum,cost_pressure,-1,0.055556,zero
gasoline_cpi_level__pct_change_jan01,economy_momentum,cost_pressure,-1,0.055556,zero
consumer_sentiment_3m__change_previous_oct31,economy_momentum,sentiment,1,0.166667,zero
unemployment_pct__change_previous_oct31,economy_momentum,employment,-1,0.166667,zero
inflation_yoy_pct__change_previous_oct31,economy_momentum,cost_pressure,-1,0.055556,zero
cpi_level__pct_change_previous_oct31,economy_momentum,cost_pressure,-1,0.055556,zero
gasoline_cpi_level__pct_change_previous_oct31,economy_momentum,cost_pressure,-1,0.055556,zero


## 4. Rerun the actual fat-tail helper

This uses the research df5 sampler, not a Gaussian interval multiplied by a constant. Its earlier architecture has no signed state factor. Eight chains run convergence checks (R-hat<1.01, bulk/tail ESS≥400), extending automatically if needed. Set RUN_STUDENT=False only to skip this slower sensitivity check.

In [6]:
if RUN_STUDENT:
    student, diagnostics = lab.run_logged(lab.rerun_student,SCENARIO,YEAR)
    d=diagnostics['diagnostics']
    display(d[['rhat','bulk_ess','tail_ess']].agg(['min','max']).round(3))
    display(student[['geography','margin_pp','p_dem','lo70_pp','hi70_pp']].round(3))
else:
    print('Student rerun skipped; saved research output remains in assets/student.')

Run log: cache/logs/rerun_student_20260921T044238.491289Z.txt


,rhat,bulk_ess,tail_ess
min,1.000,5614.46,10965.756
max,1.002,32000.00,32000.000


,geography,margin_pp,p_dem,lo70_pp,hi70_pp
0,AK,1.493,0.594,-4.964,8.077
1,AL,-18.502,0.018,-27.286,-9.705
2,AR,-11.662,0.095,-20.698,-2.454
3,CO,14.445,0.889,2.438,26.357
4,DE,25.923,0.989,16.549,35.147
5,FL,-5.289,0.219,-12.512,1.849
6,GA,6.764,0.861,0.235,13.231
7,IA,-2.601,0.310,-8.015,2.807
8,ID,-26.728,0.005,-34.367,-19.024
9,IL,22.657,0.966,11.385,34.243


## What remains uncertain

There are few independent national cycles, historical feature vintages are incomplete, and model architecture was explored repeatedly. The main covariance is learned with structural restrictions; it is not50×50 freely estimated state relationships. Candidate/caucus proxy limitations remain. This is a reproducible research model, not a claim that calibration is settled.

## Saved reports

[Read the published report](../outputs/reports/training/training.md) · [All outputs and dated reports](../outputs/README.md)
